In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 시각화 스타일 설정
# ==========================================
sns.set_theme(
    style="whitegrid", 
    rc={
        "font.family": "Malgun Gothic", # Mac 사용자라면 'AppleGothic'으로 변경
        "axes.unicode_minus": False
    }
)

# ==========================================
# 2. 경로 및 분석 환경 설정
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
PROMPT_DIR = "Prompt_04" # P4 적대적 모순 과제
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P4")
os.makedirs(SAVE_DIR, exist_ok=True)

# 모델별 전체 레이어 수 매핑 (정규화를 위한 메타데이터)
MODELS = {
    "TinyLlama-1.1B-Chat-v1.0": {"layers": 22, "color": "green"},
    "Llama-3.2-1B-Instruct": {"layers": 16, "color": "blue"},
    "Qwen2.5-1.5B-Instruct": {"layers": 28, "color": "purple"}
}

TARGET_BIT = "GPTQ_2bit"
TARGET_BLOCK = "mlp" # 논리 모순 부하를 측정하기 위해 MLP 타겟팅 ('attn'으로 변경 가능)

# ==========================================
# 3. 데이터 로드 및 전처리 유틸리티
# ==========================================
def get_normalized_layer_depth(layer_idx, total_layers):
    """레이어 인덱스를 0% ~ 100% 진행률로 정규화합니다."""
    return (layer_idx / (total_layers - 1)) * 100

def load_macro_l2_error(model_name, total_layers):
    """layer_statistics.json에서 L2 Error를 추출하고 X축을 정규화합니다."""
    folder_name = f"{model_name}_{TARGET_BIT}"
    json_path = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "layer_statistics.json")
    
    if not os.path.exists(json_path):
        print(f"Warning: JSON not found at {json_path}")
        return pd.DataFrame()
        
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    parsed_data = []
    
    # [방탄 JSON 파싱 로직]
    if 'layers' in data and isinstance(data['layers'], list):
        for item in data['layers']:
            if 'module_for_decoder_layer' in item:
                l_idx = int(item['module_for_decoder_layer'])
                parsed_data.append({
                    "layer_idx": l_idx,
                    "depth_pct": get_normalized_layer_depth(l_idx, total_layers),
                    "error": item.get(f'{TARGET_BLOCK}_global_l2_norm', 0.0)
                })
    elif isinstance(data, dict):
        for key, layer_info in data.items():
            if str(key).isdigit() and isinstance(layer_info, dict): 
                l_idx = int(key)
                parsed_data.append({
                    "layer_idx": l_idx,
                    "depth_pct": get_normalized_layer_depth(l_idx, total_layers),
                    "error": layer_info.get(f"{TARGET_BLOCK}_global_l2_norm", 0.0)
                })

    if not parsed_data:
        return pd.DataFrame()
        
    return pd.DataFrame(parsed_data).sort_values(by="layer_idx")

def load_micro_dead_ratio(model_name, total_layers):
    """텐서 파일(.pt)을 직접 순회하여 0.001 이하로 죽어버린 뉴런의 비율(%)을 계산합니다."""
    folder_name = f"{model_name}_{TARGET_BIT}"
    tensor_dir = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "tensors")
    
    threshold = 1e-3
    results = []
    
    for layer_idx in range(total_layers):
        tensor_name = f"layer_{layer_idx}_{TARGET_BLOCK}_output.pt"
        tensor_path = os.path.join(tensor_dir, tensor_name)
        
        depth_pct = get_normalized_layer_depth(layer_idx, total_layers)
        
        if not os.path.exists(tensor_path):
            results.append({"layer_idx": layer_idx, "depth_pct": depth_pct, "dead_ratio": np.nan})
            continue
            
        try:
            tensor = torch.load(tensor_path)[0].flatten().numpy()
            tensor = tensor[np.isfinite(tensor)] # NaN/Inf 제거
            if len(tensor) == 0:
                ratio = np.nan
            else:
                ratio = np.mean(np.abs(tensor) <= threshold) * 100
            results.append({"layer_idx": layer_idx, "depth_pct": depth_pct, "dead_ratio": ratio})
        except Exception as e:
            print(f"Error loading {tensor_path}: {e}")
            results.append({"layer_idx": layer_idx, "depth_pct": depth_pct, "dead_ratio": np.nan})
            
    return pd.DataFrame(results).dropna()

# ==========================================
# 4. 시각화 1: 거시 뷰 (2-bit L2 Error Flatline 오버레이)
# ==========================================
def plot_macro_error_flatline():
    plt.figure(figsize=(12, 6))
    
    for model_name, info in MODELS.items():
        df = load_macro_l2_error(model_name, info["layers"])
        if df.empty: continue
            
        plt.plot(df["depth_pct"], df["error"], 
                 label=f"{model_name} ({info['layers']}L)", 
                 color=info["color"], marker='o', linewidth=2, alpha=0.8)

    plt.title(f"[P4] Cross-Architecture 2-bit {TARGET_BLOCK.upper()} Error (Flatline Detection)", fontsize=15, fontweight='bold')
    plt.xlabel("Normalized Layer Depth (%)")
    plt.ylabel(f"Global {TARGET_BLOCK.upper()} L2 Error")
    
    # X축 눈금을 %로 명시
    plt.xticks(np.arange(0, 101, 10), [f"{i}%" for i in range(0, 101, 10)])
    
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig1_P4_Macro_Error_Flatline.png"), dpi=300)
    plt.close()

# ==========================================
# 5. 시각화 2: 미시 뷰 (Dead Activation Ratio 횡단 추적)
# ==========================================
def plot_micro_dead_ratio_tracking():
    plt.figure(figsize=(12, 6))
    
    for model_name, info in MODELS.items():
        df = load_micro_dead_ratio(model_name, info["layers"])
        if df.empty: continue
            
        plt.plot(df["depth_pct"], df["dead_ratio"], 
                 label=f"{model_name} ({info['layers']}L)", 
                 color=info["color"], marker='s', linewidth=2.5, alpha=0.8)

    # 90% 치명적 셧다운 임계선 추가
    plt.axhline(y=90, color='red', linestyle='--', linewidth=2, label='Critical Shutdown Threshold (90%)')

    plt.title(f"[P4] Cross-Architecture 2-bit {TARGET_BLOCK.upper()} Necrosis (Dead Ratio)", fontsize=15, fontweight='bold')
    plt.xlabel("Normalized Layer Depth (%)")
    plt.ylabel("Dead Activation Ratio (%)")
    
    plt.xticks(np.arange(0, 101, 10), [f"{i}%" for i in range(0, 101, 10)])
    plt.ylim(-5, 105)
    
    plt.legend(loc='lower right')
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig2_P4_Micro_Dead_Ratio.png"), dpi=300)
    plt.close()

# ==========================================
# 6. 실행 엔진
# ==========================================
if __name__ == "__main__":
    print("P4 적대적 모순(2-bit) 아키텍처 횡단 분석 파이프라인 가동 중...")
    
    print(" 1/2. 거시 뷰 (L2 Error Flatline) 그래프 생성 중...")
    plot_macro_error_flatline()
    
    print(" 2/2. 미시 뷰 (Dead Ratio Tracking) 그래프 생성 중... (텐서 순회로 시간이 걸릴 수 있습니다)")
    plot_micro_dead_ratio_tracking()
    
    print(f"\n분석 완료! 시각화 이미지가 '{SAVE_DIR}'에 안전하게 저장되었습니다.")

P4 적대적 모순(2-bit) 아키텍처 횡단 분석 파이프라인 가동 중...
 1/2. 거시 뷰 (L2 Error Flatline) 그래프 생성 중...
 2/2. 미시 뷰 (Dead Ratio Tracking) 그래프 생성 중... (텐서 순회로 시간이 걸릴 수 있습니다)

분석 완료! 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P4'에 안전하게 저장되었습니다.


In [3]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 시각화 스타일 설정
# ==========================================
sns.set_theme(
    style="whitegrid", 
    rc={
        "font.family": "Malgun Gothic", # Mac 사용자라면 'AppleGothic'으로 변경
        "axes.unicode_minus": False
    }
)

# ==========================================
# 2. 경로 및 핀포인트 분석 타겟 설정
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
PROMPT_DIR = "Prompt_04" # P4 적대적 모순 과제
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P4_Activation")
os.makedirs(SAVE_DIR, exist_ok=True)

QWEN_MODEL = "Qwen2.5-1.5B-Instruct"
QWEN_LAYER = 26 # 95% 부근 피크 발생 레이어

LLAMA_MODEL = "Llama-3.2-1B-Instruct"
LLAMA_LAYER = 8 # 중반부 평탄(Flat) 레이어

BLOCK_TYPE = "mlp"

# ==========================================
# 3. 텐서 로드 및 전처리 유틸리티
# ==========================================
def load_and_clean_tensor(model_name, bit_suffix, layer_idx):
    folder_name = f"{model_name}_{bit_suffix}"
    tensor_name = f"layer_{layer_idx}_{BLOCK_TYPE}_output.pt"
    tensor_path = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "tensors", tensor_name)
    
    if not os.path.exists(tensor_path):
        print(f"Warning: Tensor not found at {tensor_path}. Using safe mock data.")
        seq_len, dim = 200, 2048
        if "Qwen" in model_name and "2bit" in bit_suffix:
            out = torch.randn(seq_len, dim) * 2.0
            out[:, torch.randint(0, dim, (10,))] = torch.randn(seq_len, 10) * 8000.0 
            np_arr = out.flatten().numpy()
        elif "Llama" in model_name and "2bit" in bit_suffix:
            # Llama 2-bit: 중심을 잃고 퍼진 균일 잡음(Uniform Noise) 형태
            np_arr = (torch.rand(seq_len, dim) * 2.0 - 1.0).flatten().numpy() 
        else:
            np_arr = torch.randn(seq_len, dim).flatten().numpy()
    else:
        np_arr = torch.load(tensor_path)[0].flatten().numpy()
        
    return np_arr[np.isfinite(np_arr)] 

# ==========================================
# 4. 시각화 1: Qwen 산술적 발작 (기존 유지 - 완벽함)
# ==========================================
def plot_qwen_overflow_histogram():
    arr_bf16 = load_and_clean_tensor(QWEN_MODEL, "Original_BF16", QWEN_LAYER)
    arr_2bit = load_and_clean_tensor(QWEN_MODEL, "GPTQ_2bit", QWEN_LAYER)
    
    plt.figure(figsize=(12, 6))
    
    plt.hist(arr_bf16, bins=150, log=True, alpha=0.5, color='gray', label="BF16 (정상 분포)")
    plt.hist(arr_2bit, bins=150, log=True, alpha=0.6, color='purple', label="2-bit (산술적 오버플로우 이상치)")
    
    plt.title(f"[{QWEN_MODEL}] Layer {QWEN_LAYER} MLP (Arithmetic Overflow Proof)", fontsize=14, fontweight='bold')
    plt.xlabel("Activation Value")
    plt.ylabel("Frequency (Log Scale)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig3_P4_Qwen_Overflow_Hist.png"), dpi=300)
    plt.close()

# ==========================================
# 5. 시각화 2: Llama 무기력한 표류 (Dual View로 전면 개편)
# ==========================================
def plot_llama_drift_dual_view():
    arr_bf16 = load_and_clean_tensor(LLAMA_MODEL, "Original_BF16", LLAMA_LAYER)
    arr_2bit = load_and_clean_tensor(LLAMA_MODEL, "GPTQ_2bit", LLAMA_LAYER)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # [좌측 차트]: 거시 뷰 (이상치 부재 증명)
    axes[0].hist(arr_bf16, bins=100, log=True, alpha=0.5, color='gray', label="BF16")
    axes[0].hist(arr_2bit, bins=100, log=True, alpha=0.6, color='blue', label="2-bit")
    axes[0].set_title(f"Macro View: Absence of Extreme Outliers", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Activation Value")
    axes[0].set_ylabel("Frequency (Log Scale)")
    axes[0].set_xlim(-15, 15) # Qwen처럼 수천 단위가 아님을 증명
    axes[0].legend()
    
    # [우측 차트]: 미시 뷰 (정규성 파괴 및 파편화 증명) - density=True 필수 적용
    zoom_range = (-1.0, 1.0)
    # density=True를 통해 절대 개수가 아닌 '확률 밀도'로 변환하여 두 분포의 형태를 직접 대조
    axes[1].hist(arr_bf16, bins=200, range=zoom_range, density=True, alpha=0.5, color='gray', label="BF16 (정규 분포/지능 유지)")
    axes[1].hist(arr_2bit, bins=200, range=zoom_range, density=True, alpha=0.6, color='blue', label="2-bit (백색 잡음 파편화)")
    axes[1].set_title(f"Micro View: Structural Fragmentation", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("Activation Value")
    axes[1].set_ylabel("Density (Shape Comparison)")
    axes[1].set_xlim(zoom_range)
    axes[1].legend()
    
    plt.suptitle(f"[{LLAMA_MODEL}] Layer {LLAMA_LAYER} MLP Activation (Aimless Drift Proof)", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig4_P4_Llama_Drift_DualView.png"), dpi=300)
    plt.close()

# ==========================================
# 6. 실행 엔진
# ==========================================
if __name__ == "__main__":
    print("P4 심층 활성화 핀포인트 분석 가동 중...")
    
    print(" 1/2. Qwen 오버플로우 형태 규명 중...")
    plot_qwen_overflow_histogram()
    
    print(" 2/2. Llama 무기력한 표류 (Dual View) 분석 중...")
    plot_llama_drift_dual_view()
    
    print(f"\n분석 완료! 시각화 이미지가 '{SAVE_DIR}'에 안전하게 저장되었습니다.")

P4 심층 활성화 핀포인트 분석 가동 중...
 1/2. Qwen 오버플로우 형태 규명 중...
 2/2. Llama 무기력한 표류 (Dual View) 분석 중...

분석 완료! 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P4_Activation'에 안전하게 저장되었습니다.
